In [30]:
# Libraries Import
import numpy as np 
import os
import pandas as pd 
import json
import pickle
import math
from typing import Dict, Any
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [31]:
# set Path and Config
data_path = "./data/smart_home_energy_consumption.csv"
artifacts_path = "./artifacts"
os.makedirs(artifacts_path, exist_ok=True)

0. Read CSV and checking missing columns

In [32]:
expected_columns = ["Home ID", "Appliance Type", "Energy Consumption (kWh)",
    "Time", "Date", "Outdoor Temperature (°C)",
    "Season", "Household Size"]

df_raw = pd.read_csv(data_path)

missing_columns = [c for c in expected_columns if c not in df_raw.columns]

if missing_columns:
    raise ValueError(f"missing required columns {missing_columns}")

In [33]:
df = df_raw.rename(columns = {
    "Home ID" : "home_id",
    "Appliance Type" : "appliance",
    "Energy Consumption (kWh)" : "kwh",
    "Time" : "time",
    "Date" : "date",
    "Outdoor Temperature (°C)" : "temp_c",
    "Season" : "season",
    "Household Size":"hh_size"

})

In [34]:
df.head(5)

,home_id,appliance,kwh,time,date,temp_c,season,hh_size
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5


1. Type coercion


In [35]:
df["kwh"] = pd.to_numeric(df["kwh"], errors="coerce")
df["temp_c"] = pd.to_numeric(df["temp_c"], errors="coerce")
df["hh_size"] = pd.to_numeric(df["hh_size"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], format="%H:%M", errors="coerce").dt.time

In [36]:
df.dtypes

home_id               int64
appliance            object
kwh                 float64
time                 object
date         datetime64[ns]
temp_c              float64
season               object
hh_size               int64
dtype: object

In [37]:
df.shape

(100000, 8)

In [38]:
#drop unusable rows across needed columns
df = df.dropna(subset=["date", "time", "appliance", "kwh"]).copy()
df.shape

(100000, 8)

In [39]:
df.head(5)

,home_id,appliance,kwh,time,date,temp_c,season,hh_size
0,94,Fridge,0.20,21:12:00,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11:00,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39:00,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56:00,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31:00,2023-08-26,34.5,Summer,5


In [40]:
# a single appliance used by a house across different time and dates
df[(df['home_id']==94) & (df['appliance']=='Fridge')]

,home_id,appliance,kwh,time,date,temp_c,season,hh_size
0,94,Fridge,0.20,21:12:00,2023-12-02,-1.0,Fall,2
1476,94,Fridge,0.33,18:44:00,2023-05-30,27.8,Spring,4
6020,94,Fridge,0.12,03:45:00,2023-01-24,-5.8,Winter,1
7784,94,Fridge,0.21,21:27:00,2023-09-17,8.1,Summer,5
16358,94,Fridge,0.39,07:03:00,2023-11-15,23.2,Fall,5
19654,94,Fridge,0.49,22:14:00,2023-12-04,14.6,Fall,4
21888,94,Fridge,0.18,12:58:00,2023-03-01,-3.8,Winter,5
27674,94,Fridge,0.45,13:58:00,2023-05-31,9.3,Spring,5
31750,94,Fridge,0.48,21:26:00,2023-07-27,7.1,Summer,5
34154,94,Fridge,0.38,03:07:00,2023-01-27,-5.5,Winter,4


In [41]:
# combine date and time to timestamp 
df["timestamp"] = pd.to_datetime(df["date"].dt.strftime("%Y-%m-%d") + " " + df["time"].astype(str) , errors="coerce")
df["timestamp"].head(5)

0   2023-12-02 21:12:00
1   2023-08-06 20:11:00
2   2023-11-21 06:39:00
3   2023-01-21 21:56:00
4   2023-08-26 04:31:00
Name: timestamp, dtype: datetime64[ns]

In [42]:
#drop unusable rows in timestamp column
df = df.dropna(subset=["timestamp"]).copy()
df.shape

(100000, 9)

In [43]:
# sanity boundary for energy consumption
df = df[ (df["kwh"] >= 0 ) & (df["kwh"] <= 50 )].copy()
df.shape

(100000, 9)

In [44]:
# sanity boundary for household size
df = df[ ((df["hh_size"] >=1 ) & (df["hh_size"] <= 20 )) | df["hh_size"].isna()].copy()
df.shape

(100000, 9)

In [45]:
df["month"] = df["date"].dt.month

# Season imputation (if missing)
month_to_season = {
    12: "Winter",1:  "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
}

df["season"] = df["season"].fillna(df["month"].map(month_to_season))
df.shape

(100000, 10)

In [46]:
df.head(5)

,home_id,appliance,kwh,time,date,temp_c,season,hh_size,timestamp,month
0,94,Fridge,0.20,21:12:00,2023-12-02,-1.0,Fall,2,2023-12-02 21:12:00,12
1,435,Oven,0.23,20:11:00,2023-08-06,31.1,Summer,5,2023-08-06 20:11:00,8
2,466,Dishwasher,0.32,06:39:00,2023-11-21,21.3,Fall,3,2023-11-21 06:39:00,11
3,496,Heater,3.92,21:56:00,2023-01-21,-4.2,Winter,1,2023-01-21 21:56:00,1
4,137,Microwave,0.44,04:31:00,2023-08-26,34.5,Summer,5,2023-08-26 04:31:00,8


2. Aggregation

In [47]:
# daily aggregation based on per home, appliance, date

aggregated_df = df.groupby(["home_id", "appliance","date"], as_index=False).agg(daily_kwh=("kwh", "sum"),
                                                                                average_temp=("temp_c", "mean"),
                                                                                min_temp=("temp_c", "min"),
                                                                                max_temp=("temp_c", "min"),
                                                                                hh_size=("hh_size","max"),
                                                                                events=("kwh","count")
                                                                                )
aggregated_df.shape


(97422, 9)

In [48]:
aggregated_df.head(5)

,home_id,appliance,date,daily_kwh,average_temp,min_temp,max_temp,hh_size,events
0,1,Air Conditioning,2023-01-07,2.77,-3.3,-3.3,-3.3,3,1
1,1,Air Conditioning,2023-01-14,4.80,2.1,2.1,2.1,4,1
2,1,Air Conditioning,2023-01-15,3.13,1.0,1.0,1.0,2,1
3,1,Air Conditioning,2023-01-19,3.10,32.4,32.4,32.4,4,1
4,1,Air Conditioning,2023-01-29,4.16,34.6,34.6,34.6,3,1


In [49]:
aggregated_df["day_of_week"] = aggregated_df["date"].dt.dayofweek
aggregated_df["is_weekend"] = aggregated_df["day_of_week"].isin(["5", "6"]).astype(int)
aggregated_df["month"] = aggregated_df["date"].dt.month
aggregated_df["season"] = aggregated_df["month"].map(month_to_season)
aggregated_df.shape

(97422, 13)

In [50]:
aggregated_df.head(5)

,home_id,appliance,date,daily_kwh,average_temp,min_temp,max_temp,hh_size,events,day_of_week,is_weekend,month,season
0,1,Air Conditioning,2023-01-07,2.77,-3.3,-3.3,-3.3,3,1,5,0,1,Winter
1,1,Air Conditioning,2023-01-14,4.80,2.1,2.1,2.1,4,1,5,0,1,Winter
2,1,Air Conditioning,2023-01-15,3.13,1.0,1.0,1.0,2,1,6,0,1,Winter
3,1,Air Conditioning,2023-01-19,3.10,32.4,32.4,32.4,4,1,3,0,1,Winter
4,1,Air Conditioning,2023-01-29,4.16,34.6,34.6,34.6,3,1,6,0,1,Winter


3. Cross-Home Appliance-Level Daily Series (Prophet-ready)

   Aggregate across homes to learn generalized appliance behavior

In [51]:
appliance_agg_daily = aggregated_df.groupby(["appliance","date"], as_index=False).agg(y=("daily_kwh","mean"),
                                                avg_temp=("average_temp","mean"),
                                                hh_size=("hh_size","mean"),
                                                is_weekend=("is_weekend","max")).rename(columns={"date":"ds"}).sort_values(["appliance","ds"])

# drop rows lacking core columns
appliance_agg_daily = appliance_agg_daily.dropna(subset=["ds","y","avg_temp"])

In [52]:
appliance_agg_daily 

,appliance,ds,y,avg_temp,hh_size,is_weekend
0,Air Conditioning,2023-01-01,3.733448,15.896552,3.034483,0
1,Air Conditioning,2023-01-02,3.922308,17.776923,3.269231,0
2,Air Conditioning,2023-01-03,3.571905,10.247619,2.952381,0
3,Air Conditioning,2023-01-04,3.718333,14.780000,3.233333,0
4,Air Conditioning,2023-01-05,3.623750,14.743750,2.925000,0
...,...,...,...,...,...,...
3655,Washing Machine,2023-12-28,1.206957,13.878261,2.478261,0
3656,Washing Machine,2023-12-29,1.109655,15.262069,2.827586,0
3657,Washing Machine,2023-12-30,1.240000,18.755000,3.400000,0
3658,Washing Machine,2023-12-31,1.181667,11.820000,3.266667,0


 4. Time-Aware Split

In [53]:
all_dates = np.sort(appliance_agg_daily["ds"].unique())
cut_idx = int(0.8 * len(all_dates)) 
split_date = all_dates[cut_idx] if cut_idx < len(all_dates) else all_dates[-1]
print(f"Time split: Train <= {pd.to_datetime(split_date).date()} | Valid > split")

Time split: Train <= 2023-10-20 | Valid > split


5. SMAPE (Symmetric Mean Absolute Percentage Error) is a metric used to measure how accurate your predictions are—especially in forecasting problems.

🧠 What SMAPE means

SMAPE compares actual values vs predicted values and expresses the error as a percentage, but in a symmetric way (it treats over- and under-predictions more fairly than MAPE).

![alt text](image.png)

In [54]:
def smape(y_true, y_pred) -> float:
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = np.abs(y_true) + np.abs(y_pred)
    denominator = np.where(denominator == 0, 1, denominator)
    
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) / denominator
    )

6. Prophet Factory


In [55]:
def make_prophet():
    """
    Daily/weekly/yearly seasonalities with external regressors.
    """
    m = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode="additive"
        )
    m.add_regressor("avg_temp", standardize=True)
    m.add_regressor("hh_size", standardize=True)
    m.add_regressor("is_weekend")

    return m

7.  Train, Validate, Save per-appliance models

In [56]:
model_registry: Dict[str, Dict[str, Any]] = {}
scores = []
for app in sorted(appliance_agg_daily ["appliance"].unique()):
    df_app = appliance_agg_daily [appliance_agg_daily ["appliance"] == app].copy()
    if df_app["ds"].nunique() < 60:
        print(f"Skipping '{app}': insufficient history (<60 days).")

    train_df = df_app[df_app["ds"] <= split_date][["ds","y","avg_temp","hh_size","is_weekend"]].dropna()
    valid_df = df_app[df_app["ds"] >  split_date][["ds","y","avg_temp","hh_size","is_weekend"]].dropna()

    if len(valid_df) < 7:
        print(f"Skipping '{app}': too small validation window.")
        continue

    m = make_prophet()
    m.fit(train_df)

    fc_valid = m.predict(valid_df[["ds","avg_temp","hh_size","is_weekend"]])
    y_true = valid_df["y"].values
    y_pred = fc_valid["yhat"].values

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    s    = smape(y_true, y_pred)

    print(f"[{app}] N={len(y_true)}  MAE={mae:.4f} kWh  RMSE={rmse:.4f} kWh  sMAPE={s:.2f}%")
    scores.append({"appliance": app, "n": len(y_true), "mae": mae, "rmse": rmse, "smape": s})


    # Save model
    safe_app = app.replace(" ", "_").lower()
    model_path = os.path.join(artifacts_path, f"prophet_{safe_app}.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(m, f)

    # Register
    model_registry[app] = {
        "model_path": model_path,
        "regressors": ["avg_temp", "hh_size", "is_weekend"],
        "split_date": str(pd.to_datetime(split_date).date())
    }

13:36:46 - cmdstanpy - INFO - Chain [1] start processing
13:36:47 - cmdstanpy - INFO - Chain [1] done processing


[Air Conditioning] N=73  MAE=0.1572 kWh  RMSE=0.2011 kWh  sMAPE=4.41%


13:36:47 - cmdstanpy - INFO - Chain [1] start processing
13:36:47 - cmdstanpy - INFO - Chain [1] done processing


[Computer] N=73  MAE=0.1141 kWh  RMSE=0.1407 kWh  sMAPE=10.08%


13:36:47 - cmdstanpy - INFO - Chain [1] start processing
13:36:48 - cmdstanpy - INFO - Chain [1] done processing


...

[Oven] N=73  MAE=0.0862 kWh  RMSE=0.1099 kWh  sMAPE=7.73%


13:36:51 - cmdstanpy - INFO - Chain [1] start processing
13:36:51 - cmdstanpy - INFO - Chain [1] done processing


[TV] N=73  MAE=0.1015 kWh  RMSE=0.1285 kWh  sMAPE=8.94%


13:36:52 - cmdstanpy - INFO - Chain [1] start processing
13:36:52 - cmdstanpy - INFO - Chain [1] done processing


[Washing Machine] N=73  MAE=0.1221 kWh  RMSE=0.1544 kWh  sMAPE=10.99%


In [57]:
# Weighted metrics across appliances
if scores:
    total_n = sum(s["n"] for s in scores)
    w_mae   = sum(s["mae"]  * s["n"] for s in scores) / total_n
    w_rmse  = sum(s["rmse"] * s["n"] for s in scores) / total_n
    w_smape = sum(s["smape"]* s["n"] for s in scores) / total_n
    print("\n=== Weighted Validation Metrics ===")
    print(f"MAE={w_mae:.4f} kWh | RMSE={w_rmse:.4f} kWh | sMAPE={w_smape:.2f}%")



=== Weighted Validation Metrics ===
MAE=0.1208 kWh | RMSE=0.1496 kWh | sMAPE=8.96%


8. Save Clean Registry

In [58]:
REGISTRY_PATH = os.path.join(artifacts_path, "prophet_registry.json")
# Normalize paths before saving
clean_registry = {}
for app, meta in model_registry.items():
    raw_path = meta["model_path"]
    # Ensure forward slashes and remove redundant folder prefixes
    clean_path = raw_path.replace("\\", "/").replace("artifacts/", "").replace("./", "")
    clean_registry[app] = {**meta, "model_path": clean_path}

with open(REGISTRY_PATH, "w") as f:
    json.dump(clean_registry, f, indent=2)

print(f"\n✅ Saved cleaned model registry → {REGISTRY_PATH}")



✅ Saved cleaned model registry → ./artifacts\prophet_registry.json


Inference Helper 


In [ ]:
from pathlib import Path


def predict_next_day(
    appliance: str,
    ds_next: str,           # "YYYY-MM-DD"
    avg_temp: float,
    hh_size: float,
    is_weekend: int         # 0/1
) -> Dict[str, Any]:
    """
    Load the per-appliance Prophet model and forecast next-day kWh.
    Supports registry model paths saved as relative (e.g., prophet_air_conditioning.pkl)
    or absolute paths.
    """
    # Load registry
    with open(REGISTRY_PATH) as f:
        reg = json.load(f)

    if appliance not in reg:
        raise ValueError(f"No Prophet model registered for appliance: '{appliance}'")

    # Resolve path to model
    raw_path = reg[appliance]["model_path"]
    model_path = Path(raw_path)
    
    # If the model path is not absolute, assume it's relative to artifacts_path
    if not model_path.is_absolute():
        model_path = artifacts_path / model_path

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    # Load Prophet model
    with open(model_path, "rb") as f:
        m: Prophet = pickle.load(f)

    # Build future row with required regressors
    future = pd.DataFrame([{
        "ds": pd.to_datetime(ds_next),
        "avg_temp": float(avg_temp),
        "hh_size": float(hh_size),
        "is_weekend": int(is_weekend)
    }])

    fc = m.predict(future)
    print(fc)
    return {
        "appliance": appliance,
        "date": pd.to_datetime(ds_next).strftime("%Y-%m-%d"),
        "yhat": float(fc.loc[0, "yhat"]),
        "yhat_lower": float(fc.loc[0, "yhat_lower"]),
        "yhat_upper": float(fc.loc[0, "yhat_upper"])
    }